# Model V3 -- Imputing the unknowns with standard mean/mode, except for some special variables

## Data Loading and Mapping

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score
import warnings

warnings.filterwarnings('ignore')

df = pd.read_csv('filtered_data.csv')

df.head()

,_BMI5,_BMI5CAT,_AGE_G,_AGE80,_AGE65YR,_INCOMG1,INCOME3,_SMOKER3,_RFSMOK3,SMOKDAY2,...,MENTHLTH,ADDEPEV3,LANDSEX3,SEXVAR,DIABETE4,PERSDOC3,PRIMINS2,_URBSTAT,_IMPRACE,CHILDREN
0,2249.0,2.0,6.0,78.0,2.0,9.0,99.0,4.0,1.0,NaN,...,88.0,2.0,2.0,2.0,3.0,2.0,3.0,1.0,1.0,88.0
1,2583.0,3.0,6.0,80.0,2.0,7.0,11.0,3.0,1.0,3.0,...,88.0,2.0,1.0,1.0,3.0,1.0,3.0,1.0,1.0,88.0
2,2253.0,2.0,5.0,59.0,1.0,9.0,99.0,1.0,2.0,1.0,...,88.0,2.0,1.0,1.0,3.0,3.0,1.0,1.0,1.0,88.0
3,2509.0,3.0,6.0,80.0,2.0,4.0,6.0,4.0,1.0,NaN,...,88.0,2.0,1.0,1.0,3.0,1.0,3.0,1.0,1.0,88.0
4,1977.0,2.0,4.0,47.0,1.0,2.0,3.0,4.0,1.0,NaN,...,88.0,2.0,1.0,1.0,3.0,1.0,5.0,1.0,1.0,88.0


In [2]:
df.columns

Index(['_BMI5', '_BMI5CAT', '_AGE_G', '_AGE80', '_AGE65YR', '_INCOMG1',
       'INCOME3', '_SMOKER3', '_RFSMOK3', 'SMOKDAY2', 'CVDINFR4', 'CVDCRHD4',
       'ASTHMA3', '_LTASTH1', 'CHCKDNY2', 'MARITAL', 'EDUCA', '_EDUCAG',
       'GENHLTH', 'EXERANY2', '_TOTINDA', 'HAVARTH4', '_DRDXAR2', 'MENTHLTH',
       'ADDEPEV3', 'LANDSEX3', 'SEXVAR', 'DIABETE4', 'PERSDOC3', 'PRIMINS2',
       '_URBSTAT', '_IMPRACE', 'CHILDREN'],
      dtype='str')

In [3]:
chosen_columns = ['_BMI5', '_AGE_G', 'INCOME3', '_SMOKER3', 'CVDINFR4', 'CVDCRHD4',
                 'ASTHMA3', 'CHCKDNY2', 'MARITAL', 'EDUCA', 'GENHLTH', 'EXERANY2',
                 'HAVARTH4', 'MENTHLTH', 'ADDEPEV3', 'SEXVAR', 'PERSDOC3', 'PRIMINS2',
                 '_URBSTAT', 'CHILDREN', 'DIABETE4']

df_eda = df[chosen_columns]

In [4]:
df_eda.head()

,_BMI5,_AGE_G,INCOME3,_SMOKER3,CVDINFR4,CVDCRHD4,ASTHMA3,CHCKDNY2,MARITAL,EDUCA,...,EXERANY2,HAVARTH4,MENTHLTH,ADDEPEV3,SEXVAR,PERSDOC3,PRIMINS2,_URBSTAT,CHILDREN,DIABETE4
0,2249.0,6.0,99.0,4.0,2.0,2.0,2.0,2.0,3.0,4.0,...,1.0,1.0,88.0,2.0,2.0,2.0,3.0,1.0,88.0,3.0
1,2583.0,6.0,11.0,3.0,2.0,1.0,2.0,2.0,1.0,6.0,...,1.0,1.0,88.0,2.0,1.0,1.0,3.0,1.0,88.0,3.0
2,2253.0,5.0,99.0,1.0,2.0,2.0,2.0,2.0,6.0,5.0,...,1.0,1.0,88.0,2.0,1.0,3.0,1.0,1.0,88.0,3.0
3,2509.0,6.0,6.0,4.0,2.0,2.0,2.0,2.0,1.0,6.0,...,1.0,1.0,88.0,2.0,1.0,1.0,3.0,1.0,88.0,3.0
4,1977.0,4.0,3.0,4.0,2.0,2.0,2.0,2.0,5.0,5.0,...,2.0,2.0,88.0,2.0,1.0,1.0,5.0,1.0,88.0,3.0


In [5]:
df_eda['DIABETE4'] = df_eda['DIABETE4'].map({
    1: 1,          # Yes
    2: np.nan,
    3: 0,          # No
    4: 1,          # Yes (gestational)
    7: np.nan,
    9: np.nan
})

In [6]:
from sklearn.impute import SimpleImputer

# 1. Numeric columns

numeric_cols = ['CHILDREN', 'MENTHLTH']  # add any other numeric features
num_imputer = SimpleImputer(strategy='median')  # median is robust to outliers
df_eda[numeric_cols] = num_imputer.fit_transform(df_eda[numeric_cols])

In [7]:
# 2. Binary/Ordinal columns

ordinal_cols = ['_AGE_G', 'GENHLTH', 'EDUCA']  # treat as ordinal
ord_imputer = SimpleImputer(strategy='most_frequent')  # mode works, ig
df_eda[ordinal_cols] = ord_imputer.fit_transform(df_eda[ordinal_cols])

In [8]:
# 3. Nominal/Categorical columns

nominal_cols = [
    'MARITAL', 'SEXVAR', 'PERSDOC3', 'PRIMINS2',
    '_URBSTAT', 'EXERANY2', 'CVDINFR4', 'CVDCRHD4',
    'ASTHMA3', 'CHCKDNY2', 'HAVARTH4', 'ADDEPEV3'
]
nom_imputer = SimpleImputer(strategy='most_frequent')
df_eda[nominal_cols] = nom_imputer.fit_transform(df_eda[nominal_cols])


In [9]:
# 4. Make the predictive imputation method

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
import numpy as np
import pandas as pd

# Helper: predictive imputation
def predictive_impute(df, target_col, feature_cols, is_categorical=True):
    """
    Impute missing values in target_col using related features.

    Parameters:
        df: pd.DataFrame
        target_col: column to impute
        feature_cols: list of predictor columns
        is_categorical: if True, use classifier; else regressor
    """
    df_copy = df.copy()

    # Split known / unknown
    known = df_copy[df_copy[target_col].notna()]
    unknown = df_copy[df_copy[target_col].isna()]

    if unknown.shape[0] == 0:
        return df_copy[target_col]  # nothing to impute

    # Prepare features
    X_known = known[feature_cols]
    y_known = known[target_col]
    X_unknown = unknown[feature_cols]

    # Encode categoricals as integers for the model
    for col in X_known.columns:
        if X_known[col].dtype.name == 'category' or X_known[col].dtype == object:
            X_known[col] = X_known[col].astype('category').cat.codes
            X_unknown[col] = X_unknown[col].astype('category').cat.codes

    # Train model to predict missing
    if is_categorical:
        model = RandomForestClassifier(n_estimators=100, random_state=42)
    else:
        model = RandomForestRegressor(n_estimators=100, random_state=42)

    model.fit(X_known, y_known)

    # Predict missing values
    imputed_vals = model.predict(X_unknown)
    df_copy.loc[df_copy[target_col].isna(), target_col] = imputed_vals

    return df_copy[target_col]


In [10]:
# 5. Apply predictive imputation
# INCOME3 -> categorical
df_eda['INCOME3'] = predictive_impute(
    df_eda, 'INCOME3',
    ['PRIMINS2', 'EDUCA', 'MARITAL', 'GENHLTH', '_AGE_G', 'EXERANY2', 'SEXVAR', '_URBSTAT'],
    is_categorical=True
)

# _BMI5 -> numeric
df_eda['_BMI5'] = predictive_impute(
    df_eda, '_BMI5',
    ['GENHLTH', 'EXERANY2', 'HAVARTH4', '_AGE_G', 'ASTHMA3', '_SMOKER3'],
    is_categorical=False
)

# _SMOKER3 -> categorical
df_eda['_SMOKER3'] = predictive_impute(
    df_eda, '_SMOKER3',
    ['_AGE_G', 'EDUCA', 'ADDEPEV3', 'SEXVAR', 'HAVARTH4'],
    is_categorical=True
)

In [11]:
# 6. Check if any NaNs remain
# -----------------------------
print("Remaining NaNs per column:\n", df_eda.isna().sum())

Remaining NaNs per column:
 _BMI5          0
_AGE_G         0
INCOME3        0
_SMOKER3       0
CVDINFR4       0
CVDCRHD4       0
ASTHMA3        0
CHCKDNY2       0
MARITAL        0
EDUCA          0
GENHLTH        0
EXERANY2       0
HAVARTH4       0
MENTHLTH       0
ADDEPEV3       0
SEXVAR         0
PERSDOC3       0
PRIMINS2       0
_URBSTAT       0
CHILDREN       0
DIABETE4    4429
dtype: int64


In [12]:
df_eda.head()

,_BMI5,_AGE_G,INCOME3,_SMOKER3,CVDINFR4,CVDCRHD4,ASTHMA3,CHCKDNY2,MARITAL,EDUCA,...,EXERANY2,HAVARTH4,MENTHLTH,ADDEPEV3,SEXVAR,PERSDOC3,PRIMINS2,_URBSTAT,CHILDREN,DIABETE4
0,2249.0,6.0,99.0,4.0,2.0,2.0,2.0,2.0,3.0,4.0,...,1.0,1.0,88.0,2.0,2.0,2.0,3.0,1.0,88.0,0.0
1,2583.0,6.0,11.0,3.0,2.0,1.0,2.0,2.0,1.0,6.0,...,1.0,1.0,88.0,2.0,1.0,1.0,3.0,1.0,88.0,0.0
2,2253.0,5.0,99.0,1.0,2.0,2.0,2.0,2.0,6.0,5.0,...,1.0,1.0,88.0,2.0,1.0,3.0,1.0,1.0,88.0,0.0
3,2509.0,6.0,6.0,4.0,2.0,2.0,2.0,2.0,1.0,6.0,...,1.0,1.0,88.0,2.0,1.0,1.0,3.0,1.0,88.0,0.0
4,1977.0,4.0,3.0,4.0,2.0,2.0,2.0,2.0,5.0,5.0,...,2.0,2.0,88.0,2.0,1.0,1.0,5.0,1.0,88.0,0.0


In this first version, we try to just categorize the "Don't know" and "Refused" answers to be unknown since they are still important data to keep that they're considered Missing Not at Random(MNAR), not Missing Completely at Random(MCAR)

## Data Cleaning and Splitting

In [13]:
# Drop rows where DIABETE4 is NaN
df_eda = df_eda.dropna(subset=['DIABETE4'])

# Reset the index so it’s clean
df_eda = df_eda.reset_index(drop=True)

# Quick check
df_eda['DIABETE4'].value_counts(dropna=False)

DIABETE4
0.0    376125
1.0     77116
Name: count, dtype: int64

In [14]:
from sklearn.model_selection import train_test_split

# Features (X) and target (y)
X = df_eda.drop(columns=['DIABETE4'])
y = df_eda['DIABETE4']

# 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=67, stratify=y
)

# Quick check
print("Train shape:", X_train.shape)
print("Test shape: ", X_test.shape)
print("Train target distribution:\n", y_train.value_counts(normalize=True))
print("Test target distribution:\n", y_test.value_counts(normalize=True))

Train shape: (362592, 20)
Test shape:  (90649, 20)
Train target distribution:
 DIABETE4
0.0    0.829856
1.0    0.170144
Name: proportion, dtype: float64
Test target distribution:
 DIABETE4
0.0    0.82986
1.0    0.17014
Name: proportion, dtype: float64


## Data Train and Testing

In [15]:
import xgboost as xgb
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, f1_score
import numpy as np

# 1. Compute scale_pos_weight
pos = y_train.sum()
neg = len(y_train) - pos
scale = neg / pos

In [16]:
# 2. Create XGBoost model
model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    missing=np.nan,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss',
    scale_pos_weight=scale
)

In [17]:
# 3. Train the model
model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

In [18]:
# 4. Predict probabilities
y_proba = model.predict_proba(X_test)[:,1]

In [19]:
# # # 5a. Threshold tuning for best F1
# thresholds = np.linspace(0.1, 0.9, 81)
# f1_scores = []
#
# for t in thresholds:
#     y_pred_t = (y_proba > t).astype(int)
#     f1_scores.append(f1_score(y_test, y_pred_t))
#
# best_idx = np.argmax(f1_scores)
# best_threshold = thresholds[best_idx]
# y_pred_best = (y_proba > best_threshold).astype(int)
#
# print(f"Best threshold for max F1: {best_threshold:.2f}")
# print(f"Best F1-score: {f1_scores[best_idx]:.4f}")


In [20]:
# 5b. Threshold tuning for best F2
from sklearn.metrics import fbeta_score

thresholds = np.linspace(0.1, 0.9, 81)
f2_scores = []

for t in thresholds:
    y_pred_t = (y_proba > t).astype(int)
    f2_scores.append(fbeta_score(y_test, y_pred_t, beta=2))

best_idx = np.argmax(f2_scores)
best_threshold = thresholds[best_idx]
y_pred_best = (y_proba > best_threshold).astype(int)

print(f"Best threshold for max F2: {best_threshold:.2f}")
print(f"Max F2-score: {f2_scores[best_idx]:.4f}")

Best threshold for max F2: 0.36
Max F2-score: 0.6173


In [21]:
from sklearn.metrics import confusion_matrix
import pandas as pd

# Evaluate
accuracy = accuracy_score(y_test, y_pred_best)
roc_auc = roc_auc_score(y_test, y_proba)
report = classification_report(y_test, y_pred_best, output_dict=True)

# Convert classification report to DataFrame for nicer print
report_df = pd.DataFrame(report).transpose()

# Confusion matrix for quick glance
cm = confusion_matrix(y_test, y_pred_best)

print("Model Evaluation")
print(f"Accuracy : {accuracy:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}\n")

print("---- Confusion Matrix ----")
print(pd.DataFrame(cm, index=['Actual 0','Actual 1'], columns=['Pred 0','Pred 1']))
print("\n---- Classification Report ----")
# Round numbers for cleaner look
print(report_df[['precision','recall','f1-score']].round(2))

Model Evaluation
Accuracy : 0.5882
ROC-AUC  : 0.7958

---- Confusion Matrix ----
          Pred 0  Pred 1
Actual 0   39619   35607
Actual 1    1719   13704

---- Classification Report ----
              precision  recall  f1-score
0.0                0.96    0.53      0.68
1.0                0.28    0.89      0.42
accuracy           0.59    0.59      0.59
macro avg          0.62    0.71      0.55
weighted avg       0.84    0.59      0.64
